# Agentic Search with Shell tool

This notebook walks you through an agentic search example with a shell tool that lets the agent run commands in the terminal to navigate a local filesystem and use `grep` to search files. 

It builds on the first implementation by adding `jina-grep-cli` as a semantic search alternative to `grep`.

![](../img/agent_search_for_context_engineering_overview.png)

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

LITELLM_API_KEY = os.getenv("LITELLM_API_KEY")
LITELLM_API_BASE = os.getenv("LITELLM_API_BASE")

## Set up agent with shell tool (and `grep`)

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    model="llm-gateway/gpt-5.4-nano",
    temperature=0.5,
)

### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to a **local filesystem** with the conference data.

In [14]:
SYSTEM_PROMPT = open("system_prompt_shell.md", 'r').read()

print(SYSTEM_PROMPT)

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Local filesystem (`session_data`)

The conference sessions are available under under `../data/session_data/`. One file per session.

File structure:
```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── ...
  └── expo_session/
```

File Sample:
```
# <Title>

- **Day:** <Date of the session>
- **Time:** <Time slot of the session>
- **Room:** <Room where the session takes place>
- **Type:** <One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session'>
- **Speakers:** <Name(s) of the speaker(s)>

<Description>
``

### Create tools

The LLM can use it to execute any shell commands. A common use case for this is letting the LLM interact with your local file system.

[Shell Tool](https://docs.langchain.com/oss/python/integrations/tools/bash)

<div class="alert alert-block alert-warning">
Giving agents access to the shell is powerful (though risky outside a sandboxed environment).
Note: Shell tool does not work with Windows OS.
The shell tool has no safeguards by default. Use at your own risk.
</div>

In [15]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

# Test the shell tool
print(shell_tool.run({"commands": ["echo 'Hello World!'"]}))

Executing command:
 ["echo 'Hello World!'"]
Hello World!



/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### Create the agent

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

## Run the agent

In [17]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "Which sessions should I visit to learn more about using Codex?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
        stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions should I visit to learn more about using Codex?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_NSrhILv4qUnOo2UzCqZ398l5)
 Call ID: call_NSrhILv4qUnOo2UzCqZ398l5
  Args:
    commands: ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "Codex" ../data/session_data | head -n 50']
Executing command:
 ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "Codex" ../data/session_data | head -n 50']
================================= Tool Message =================================
Name: terminal

expo_session
keynote
lightning
talk
track_keynote
workshop

../data/session_data/expo_session:
Expo Session – Arize #1.txt
Expo Session – Arize #1__2.txt
     162
../data/session_data/workshop/Codex and Subagents.txt
../data/session_data/talk/The Year of Latency D

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_XMLoQKBe5otjIAagCHg26dmc)
 Call ID: call_XMLoQKBe5otjIAagCHg26dmc
  Args:
    commands: ['cat "../data/session_data/workshop/Codex and Subagents.txt"', 'grep -Ril "Codex" ../data/session_data/* 2>/dev/null || true', 'grep -Ril "codex" ../data/session_data 2>/dev/null | head -n 50']
Executing command:
 ['cat "../data/session_data/workshop/Codex and Subagents.txt"', 'grep -Ril "Codex" ../data/session_data/* 2>/dev/null || true', 'grep -Ril "codex" ../data/session_data 2>/dev/null | head -n 50']
================================= Tool Message =================================
Name: terminal

# Codex and Subagents

- **Day:** April 8
- **Time:** 9:00-10:20am
- **Room:** Westminster
- **Type:** workshop
- **Speakers:** Vaibhav Srivastav
**Description:**../data/session_data/talk/The Year of Latency Debt.txt
../data/session_data/talk/Your Coding Agent Should Do AI System Engineering.tx

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

To learn more about using **Codex**, I found this session you should visit:

- **“Codex and Subagents”** (Workshop)  
  - **Day:** April 8  
  - **Time:** 9:00–10:20am  
  - **Room:** Westminster  
  - **Speaker:** Vaibhav Srivastav  
  - **Description:** Covers using **Codex** in combination with **subagents**.


In [18]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "Which sessions should I visit to learn more about using Codex?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
        stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions should I visit to learn more about using Codex?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_YfEjGjoD8Rusp2BnGIkwxG1W)
 Call ID: call_YfEjGjoD8Rusp2BnGIkwxG1W
  Args:
    commands: ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "Codex" ../data/session_data | head -n 20', 'grep -Ril "codex" ../data/session_data | head -n 20']
Executing command:
 ['ls -R ../data/session_data | head', 'find ../data/session_data -type f | wc -l', 'grep -Ril "Codex" ../data/session_data | head -n 20', 'grep -Ril "codex" ../data/session_data | head -n 20']
================================= Tool Message =================================
Name: terminal

expo_session
keynote
lightning
talk
track_keynote
workshop

../data/session_data/expo_session:
Expo Session – Arize #1.txt
Expo Session – Arize #1__2.txt


/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_usk5jnWgskGkgZrXTfssBEiN)
 Call ID: call_usk5jnWgskGkgZrXTfssBEiN
  Args:
    commands: ['sed -n \'1,120p\' "../data/session_data/workshop/Codex and Subagents.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/Your Coding Agent Should Do AI System Engineering.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/The Year of Latency Debt.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/Let\'s Talk About FOMAT (Fear of Missing Agent Time).txt"']
Executing command:
 ['sed -n \'1,120p\' "../data/session_data/workshop/Codex and Subagents.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/Your Coding Agent Should Do AI System Engineering.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/The Year of Latency Debt.txt"', 'sed -n \'1,120p\' "../data/session_data/talk/Let\'s Talk About FOMAT (Fear of Missing Agent Time).txt"']
================================= Tool Message =====

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

To learn more about using **Codex**, you should visit these sessions:

1) **“Codex and Subagents”** (Workshop)  
- **Day/Time:** April 8, 9:00–10:20am  
- **Room:** Westminster  
- **Speaker:** Vaibhav Srivastav  

2) **“Your Coding Agent Should Do AI System Engineering”** (Talk, Track: Coding Agents)  
- **Day/Time:** April 10, 2:30–2:50pm  
- **Room:** Fleming  
- **Speaker:** Ben Burtenshaw  
- **Why it’s relevant:** Includes Codex (along with Claude Code and Gemini CLI) and shows an agent-driven workflow for building things like CUDA kernels and running ML experiments.

3) **“Let’s Talk About FOMAT (Fear of Missing Agent Time)”** (Talk, Track: Coding Agents)  
- **Day/Time:** April 10, 2:50–3:10pm  
- **Room:** Fleming  
- **Speaker:** Michael Richman  
- **Why it’s relevant:** Discusses operational patterns and tools that work with **Codex CLI** (plus others), including monitoring/unblocking agents w

In [19]:
# Run the agent
input_message = {
    "role": "user",
    "content": (
        "Which sessions discuss handling regulatory constraints?"
    ),
}

# Stream
for step in agent.stream(
    {"messages": [input_message]},
        stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_UXozNon4nHOFGEpRzptd2med)
 Call ID: call_UXozNon4nHOFGEpRzptd2med
  Args:
    commands: ['ls -R ../data/session_data | head', "grep -Rin --include='*.txt' -E 'regulat|constraint|compliance|GDPR|legal|policy|governance|license|licensing|regulator' ../data/session_data/session_data 2>/dev/null || true", "grep -Rin --include='*.txt' -E 'regulat|constraint|compliance|GDPR|legal|policy|governance|license|licensing|regulator' ../data/session_data 2>/dev/null || true"]
Executing command:
 ['ls -R ../data/session_data | head', "grep -Rin --include='*.txt' -E 'regulat|constraint|compliance|GDPR|legal|policy|governance|license|licensing|regulator' ../data/session_data/session_data 2>/dev/null || true", "grep -Rin --include='*.txt' -E 'regulat|constrain

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

Sessions that discuss handling regulatory constraints:

- **Engineering AI Under Sovereignty Constraints** (talk) — explicitly frames sovereignty and **regulatory/jurisdictional constraints** as requirements that shape AI system architecture (e.g., data localization, auditability, model transparency, governance by design).
- **What Does Multi-Modal and Coding Have to Do with Legal?** (lightning) — focuses on applying multimodal + coding approaches to the **legal industry**, including how legal data can be handled for LLM use.
- **Why AI Fails in Legacy Modernisation - And How Context Makes It Work** (talk) — notes that legacy enterprises include **regulatory constraints** and emphasizes context-aware AI to avoid failures at scale.


## Set up agent with shell tool (and semantic `grep` alternative)

`grep` is based on regular expressions and is great for pattern matching. However, it has the limitation that it's not great at semantic search.

There are many alternatives to `grep` available that enable semantic search over files, for example [`jina-grep-cli`](https://github.com/jina-ai/jina-grep-cli).

### 1. Install the `jina-grep` CLI

Install the `jina-grep` CLI in your virtual environment:

```bash
git clone https://github.com/jina-ai/jina-grep-cli.git && cd jina-grep-cli
uv pip install -e .
```

### 2. Adjust the agent's system prompt

Next, you need to tell your agent about the new CLI it can use in its system prompt. Make sure to also add some explanations on how to use the `jina-grep` CLI.

In [20]:
SYSTEM_PROMPT = open("system_prompt_shell_with_jina_grep.md", 'r').read()

print(SYSTEM_PROMPT)

You are a search agent tasked with answering questions using different information sources.
Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
After you have retrieved information, check if the information is helpful for answering the question.

## Context sources

You have access to files containing session information for the AI Engineer Conference under `../data/session_data/`.

### File structure

```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── talk/
  ├── keynote/ 
  ├── track_keynote/ 
  ├── lightning/
  └── expo_session/
```

### File Sample

```
# Gemma, DeepMind's Family of Open Models

- **Day:** April 10
- **Time:** 9:00-9:20am
- **Room:** Keynote
- **Type:** keynote
- **Speakers:** Omar Sanseviero

Google DeepMind’s Gemma family is expanding. Join us for a deep dive into the latest models of the Gemma ecosystem. From vibe fine-tuning to Sovereign AI, you'll learn

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

In [22]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_Y488fFPT4470igUU4KzRRWbo)
 Call ID: call_Y488fFPT4470igUU4KzRRWbo
  Args:
    commands: ['jina-grep -r --top-k 10 --threshold 0.35 "regulatory constraints" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "regulation" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "compliance" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "legal" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "privacy" ../data/session_data']
Executing command:
 ['jina-grep -r --top-k 10 --threshold 0.35 "regulatory constraints" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "regulation" ../data/session_data', 'jina-grep -r --top-k 10 --threshold 0.35 "compliance" ../data/session_data', 'j

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================= Tool Message =================================
Name: terminal

Loading model... done
../data/session_data/talk/Engineering AI Under Sovereignty Constraints.txt:1:# Engineering AI Under Sovereignty Constraints
- **Day:** April 9
- **Time:** 12:40-1:00pm
- **Room:** St. James
- **Type:** talk
- **Track:** Context Engineering
- **Speakers:** Bilge Yücel
**Description:**
Regulatory and jurisdictional constraints are no longer an edge case in AI system design; they now influence architecture decisions as much as model quality. From European Parliament discussions around prioritising non-US technology in public procurement and building a “Eurostack,” to the launch of the AWS European Sovereign Cloud, sovereignty is becoming a concrete technical requirement.
AI systems increasingly need to operate within defined legal and physical boundaries. For engineers, this raises a practical question: **what changes when your AI system is not allowed to leave a jurisdic

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================= Tool Message =================================
Name: terminal

Loading model... done
Loading model... done
Loading model... done

================================== Ai Message ==================================

Sessions that discuss handling regulatory constraints / compliance-like requirements include:

- **Engineering AI Under Sovereignty Constraints** (Talk) — *Apr 9, 12:40–1:00pm, St. James* — Speaker: **Bilge Yücel**  
  *Focus:* designing AI systems under **regulatory and jurisdictional constraints** (e.g., data localization, auditability, transparency, governance by design).

- **What Does Multi-Modal and Coding Have to Do with Legal?** (Talk) — *Apr 9, 12:40–1:00pm, Abbey* — Speaker: **Alva Liu**  
  *Focus:* applications of multimodal/coding approaches in the **legal industry** (regulatory context in general legal workflows).
